# Clase 11 — Modelos generativos modernos: autoencoders variacionales

En la clase diez construiste un autoencoder que comprime una imagen en un vector y la reconstruye. En esta clase das el siguiente paso: conviertes ese vector fijo en una distribución de probabilidad y obtienes un modelo generativo real, el autoencoder variacional (VAE, por sus siglas en inglés, *Variational Autoencoder*). Ejecutas cada celda en orden, en Google Colab, y ves el modelo entrenar y generar dígitos nuevos en vivo.

## Objetivos de aprendizaje

- Explicas la diferencia entre el espacio latente de un autoencoder normal y el de un autoencoder variacional
- Implementas el truco de reparametrización para muestrear de una gaussiana de forma diferenciable
- Construyes y entrenas un VAE pequeño sobre el conjunto de dígitos manuscritos MNIST
- Calculas la función de pérdida ELBO (límite inferior de la evidencia, *Evidence Lower Bound*), que combina reconstrucción y divergencia de Kullback-Leibler (KL)
- Generas dígitos nuevos muestreando puntos al azar del espacio latente y decodificándolos

## De un punto a una distribución

El autoencoder de la clase anterior asigna a cada imagen un único punto fijo del espacio latente. Ese punto no te garantiza nada sobre sus vecinos: dos imágenes parecidas pueden acabar lejos entre sí, y un punto intermedio entre dos codificaciones puede no significar nada cuando lo decodificas.

El autoencoder variacional cambia la regla. En lugar de un punto, el codificador produce los parámetros de una distribución gaussiana: una media (mu) y una varianza (sigma al cuadrado) para cada imagen. El entrenamiento empuja además todas esas distribuciones a parecerse a una gaussiana estándar centrada en cero, lo que ordena el espacio latente y elimina los huecos vacíos. El resultado es un mapa continuo: cualquier punto que elijas, esté o no cerca de un dato de entrenamiento, decodifica en algo con sentido.

## El truco de reparametrización

Para entrenar el codificador con descenso de gradiente, necesitas propagar el error a través del paso de muestreo. Muestrear directamente de una gaussiana con media y varianza aprendidas bloquea el gradiente: no existe una derivada bien definida de "un número elegido al azar" respecto a los parámetros que definen ese azar.

La solución separa el azar de los parámetros entrenables. Generas un ruido fijo (epsilon) desde una gaussiana estándar, que no depende de ningún parámetro, y lo combinas con mu y sigma mediante una fórmula diferenciable:

z = mu + sigma · epsilon

El gradiente atraviesa esa fórmula sin problema, porque es una simple multiplicación y una suma. El azar queda aislado, fuera del camino de aprendizaje.

In [ ]:
import torch

# mu y log-varianza, como si vinieran de la salida de un codificador
mu = torch.tensor([0.5, -1.2], requires_grad=True)
logvar = torch.tensor([0.1, -0.3], requires_grad=True)

# sigma siempre positiva, aunque logvar pueda ser cualquier número real
sigma = torch.exp(0.5 * logvar)

# epsilon: ruido fijo, sin gradiente, generado aparte de mu y sigma
epsilon = torch.randn_like(sigma)

# truco de reparametrización: z es una función diferenciable de mu y sigma
z = mu + sigma * epsilon
print("z:", z)

# comprueba que el gradiente fluye hasta mu y logvar
z.sum().backward()
print("gradiente respecto a mu:", mu.grad)
print("gradiente respecto a logvar:", logvar.grad)

## Construye un VAE

Ahora juntas las dos piezas en un modelo completo. El codificador toma una imagen de 28 por 28 píxeles del conjunto MNIST y produce mu y log-varianza. El muestreo con reparametrización obtiene un punto z del espacio latente. El decodificador reconstruye la imagen a partir de z. Entrenas el conjunto con la pérdida ELBO: reconstrucción más divergencia KL respecto al prior.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# MNIST: dígitos manuscritos de 28x28 píxeles en escala de grises, valores en [0, 1]
transform = transforms.ToTensor()
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_data = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

LATENT_DIM = 20  # dimensión del espacio latente z

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=LATENT_DIM):
        super().__init__()
        # codificador: capa oculta compartida
        self.encoder_hidden = nn.Linear(input_dim, hidden_dim)
        # dos cabezas separadas: una produce mu, otra produce log-varianza
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # decodificador: del espacio latente de vuelta a los píxeles
        self.decoder_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder_out = nn.Linear(hidden_dim, input_dim)

    def encode(self, x):
        h = F.relu(self.encoder_hidden(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # z = mu + sigma * epsilon, con epsilon ~ N(0, I)
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps

    def decode(self, z):
        h = F.relu(self.decoder_hidden(z))
        # sigmoid: el decodificador modela una Bernoulli por píxel
        return torch.sigmoid(self.decoder_out(h))

    def forward(self, x):
        mu, logvar = self.encode(x.view(x.size(0), -1))
        z = self.reparameterize(mu, logvar)
        x_reconstruida = self.decode(z)
        return x_reconstruida, mu, logvar


modelo = VAE().to(device)
print(modelo)

In [ ]:
def elbo_loss(x_reconstruida, x, mu, logvar):
    # término de reconstrucción: log-verosimilitud de una Bernoulli, sumada sobre píxeles
    reconstruccion = F.binary_cross_entropy(
        x_reconstruida, x.view(x.size(0), -1), reduction="sum"
    )
    # divergencia KL entre q(z|x) = N(mu, sigma^2) y el prior p(z) = N(0, I), forma cerrada
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return reconstruccion + kl  # minimizar esto equivale a maximizar el ELBO

In [ ]:
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)

EPOCHS = 10
for epoch in range(1, EPOCHS + 1):
    modelo.train()
    perdida_total = 0.0
    for x, _ in train_loader:  # las etiquetas no se usan: el VAE es no supervisado
        x = x.to(device)
        optimizador.zero_grad()
        x_reconstruida, mu, logvar = modelo(x)
        perdida = elbo_loss(x_reconstruida, x, mu, logvar)
        perdida.backward()
        optimizador.step()
        perdida_total += perdida.item()

    print(f"Época {epoch}/{EPOCHS} — pérdida media: {perdida_total / len(train_data):.2f}")

In [ ]:
# comprobación rápida: ¿reconstruye bien el modelo dígitos que no vio como test?
modelo.eval()
with torch.no_grad():
    x_test, _ = next(iter(test_loader))
    x_test = x_test.to(device)
    x_reconstruida, _, _ = modelo(x_test)

n = 8
fig, ejes = plt.subplots(2, n, figsize=(n * 1.2, 3))
for i in range(n):
    ejes[0, i].imshow(x_test[i].cpu().squeeze(), cmap="gray")
    ejes[0, i].axis("off")
    ejes[1, i].imshow(x_reconstruida[i].cpu().view(28, 28), cmap="gray")
    ejes[1, i].axis("off")
plt.suptitle("Fila superior: dígitos originales — fila inferior: reconstrucciones del VAE")
plt.show()

## Genera dígitos nuevos

El decodificador ya sabe convertir un punto del espacio latente en una imagen. Ahora prescindes por completo del codificador: tomas puntos al azar directamente del prior, la gaussiana estándar N(0, I), y se los pasas al decodificador. Si el espacio latente quedó bien ordenado durante el entrenamiento, cada punto produce un dígito reconocible, sin que ninguna imagen real haya intervenido en el proceso.

In [ ]:
modelo.eval()
with torch.no_grad():
    n_muestras = 64
    z_aleatorio = torch.randn(n_muestras, LATENT_DIM).to(device)  # muestreo directo del prior N(0, I)
    muestras = modelo.decode(z_aleatorio).cpu()

filas, columnas = 8, 8
fig, ejes = plt.subplots(filas, columnas, figsize=(columnas * 1.2, filas * 1.2))
for i, ax in enumerate(ejes.flat):
    ax.imshow(muestras[i].view(28, 28), cmap="gray")
    ax.axis("off")
plt.suptitle("Dígitos generados desde puntos aleatorios del espacio latente")
plt.show()

## Resumen de la clase

- Un autoencoder normal asigna a cada dato un punto fijo del espacio latente; un VAE le asigna una distribución de probabilidad, y eso convierte el espacio latente en un mapa continuo
- El truco de reparametrización separa el azar (epsilon) de los parámetros entrenables (mu y sigma), lo que permite propagar gradientes a través del muestreo: z = mu + sigma · epsilon
- El codificador de un VAE produce dos vectores, mu y log-varianza, en lugar de un único vector de código
- La pérdida ELBO combina un término de reconstrucción (qué tan bien reconstruye el decodificador la entrada) y un término de divergencia KL (qué tan cerca está la distribución del codificador del prior gaussiano)
- Generar datos nuevos es tan simple como muestrear del prior N(0, I) y decodificar, sin necesidad de partir de ninguna imagen real

**Abre ahora el notebook de práctica companion, `../notebooks_practica/clase11_practica_modelos_generativos_modernos.ipynb`, y experimenta con el espacio latente por tu cuenta.**